# scClone2DR Tutorial

This notebook demonstrates how to use the scClone2DR package for analyzing single-cell drug response data. It covers both real data analysis and simulated data experiments.

## Prerequisites: Generate Test Data

**Important**: Before running this tutorial, you must first generate synthetic test data by running the notebook:

```
./notebooks/generate_fake_data.ipynb
```

This will create fake data that mimics the structure of the real datasets used in the paper (which are confidential and cannot be shared). The synthetic data includes:
- Single-cell RNA expression data with clone annotations
- Fast Drug response measurements
- Clone metadata

## Tutorial Contents

This notebook is divided into two main sections:

### 1. Real Data Analysis
Demonstrates the complete workflow using real-world data format:
- Loading single-cell RNA and drug response data
- Training the scClone2DR model
- Making predictions on held-out test data
- Visualizing results (fold changes, cell counts, survival probabilities)
- Analyzing clone proportions and drug effects

### 2. Simulated Data Analysis
Shows how to work with fully synthetic data where ground truth is known and where (contrary to 1.) we consider feature directly at the clone level (not at the single cell level):
- Generating simulated training data with known parameters
- Training and evaluating model performance
- Comparing predictions against ground truth
- Comprehensive result visualization

## Quick Start

1. Run `./notebooks/generate_fake_data.ipynb` to generate simulated data
2. Execute cells sequentially in this notebook
3. Adjust parameters (train/test split, regularization, training steps) as needed

In [ ]:
import sys
import scclone2dr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy

# Real Data

## Initialize Real-Data Model
Set file paths and create the scClone2DR model instance for real data.

In [ ]:
path_rna = "./data/"
path_fastdrug = "./data/FD_data.csv"
datamodule = scclone2dr.data.RealData(path_fastdrug=path_fastdrug, path_rna=path_rna)

In [ ]:
data_ref = datamodule.get_real_data(concentration_DMSO=5, concentration_drug=5)

## Train/Test Split and Training
Split the real dataset and train the model with L1/L2 regularization.

In [ ]:
idxs_train = [i for i in range(int(0.7*data_ref['N']))]
idxs_test = [i for i in range(data_ref['N']) if not(i in idxs_train)]

data_train, data_test, sample_names_train, sample_names_test = datamodule.get_real_data_split(idxs_train, idxs_test)

In [ ]:
from pathlib import Path
import numpy as np
from scclone2dr.pipeline import scClone2DRPipeline
from scclone2dr.trainer import Trainer, GuideType

# 1) Build the pipeline from the already-loaded real-data module.
trainer = Trainer(guide_type=GuideType.LOWRANK_MVN, rank=10)
pipeline = scClone2DRPipeline(
    data_source=datamodule,
    trainer=trainer,
    mode_nu="noise_correction",
    mode_theta="not shared decoupled",
)

# Ensure model topology is configured from the dataset metadata.
pipeline.model.configure(datamodule)

# 2) Fit on the already prepared data_train dictionary.
params_svi = pipeline.fit(
    data=data_train,
    penalty_l1=0.1,
    penalty_l2=0.1,
    lr=0.01,
    n_steps=2000,
)

# 3) Save learned parameters.
ckpt_dir = Path("./checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / "real_data_pipeline_run.npz"
pipeline.save(ckpt_path)

# 4) Load trained pipeline from disk.
pipeline_loaded = scClone2DRPipeline.from_file(
    ckpt_path,
    data_source=datamodule,
)
pipeline_loaded.model.configure(datamodule)

# 5) Sample from posterior (Monte Carlo).
posterior_results = pipeline_loaded.sample_posterior(
    data=data_test,
    idxs_sample_eval=idxs_test,
    nb_ites=100,
    dir_save=None,
    sample_names=sample_names_test,
    model_name="real_data_",
)

# 6) Optional posterior predictive sample from the generative model.
posterior_predictive_data, _ = pipeline_loaded.model.sampling(data_test, params=posterior_results['params'])

print(f"Saved checkpoint: {ckpt_path}")

## Compute evaluation metrics

In [ ]:
evaluations = pipeline_loaded.evaluate(data_test, posterior_results['params'])

## Fold Change Scatter Plot
Compare predicted vs observed fold changes visually.

In [ ]:
plt.scatter(evaluations.fold_change_data, evaluations.fold_change_pred)

## Fraction Visualization
Show tumor fractions for the validation data.

In [ ]:
scclone2dr.plots.show_fractions(data_test, posterior_results['data'], idxdrug=0)

## Cell Count Visualization
Display predicted number of non-malignant cells in control wells.

In [ ]:
scclone2dr.plots.show_cells(data_test, posterior_results['data'])

## Proportion Visualization
Plot clone proportions inferred by the model.

In [ ]:
scclone2dr.plots.show_proportions(data_test, posterior_results['params'])

## Beta Effects
Inspect beta parameters to interpret drug effects.

In [ ]:
scclone2dr.plots.show_beta(data_test, posterior_results['params'])

## Count Scatter Plot
Visualize observed vs predicted counts.

In [ ]:
scclone2dr.plots.scatter_counts(data_test, posterior_results['data'])

## Survival Probabilities
Compute survival probabilities from single-cell features and plot them.

In [ ]:
scclone2dr.plots.survival_probabilities(data_test, posterior_results['params']['PI'].detach().numpy(), datamodule.cluster2clonelabel, idxdrug=0)

# Simulated data

In [ ]:
import sys
import scclone2dr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy

## Generate Simulated Data
Create synthetic training data with known parameters.

In [ ]:
datamodule = scclone2dr.data.SimulatedData()
data_ref, gt_params = datamodule.get_simulated_training_data({'C':24,'R':5,'N':100,'Kmax':7, 'D':30, 'theta_rna':15}, neg_bin_n=100)

## Simulated Train/Test Split
Split simulated data and train the model.

In [ ]:
idxs_train = [i for i in range(int(0.7*data_ref['N']))]
idxs_test = [i for i in range(data_ref['N']) if not(i in idxs_train)]

data_train, data_test = datamodule.get_data_split(data_ref, idxs_train, idxs_test)
gt_params_train, gt_params_test = datamodule.get_params_split(gt_params, idxs_train, idxs_test)

In [ ]:
from pathlib import Path
import numpy as np
from scclone2dr.pipeline import scClone2DRPipeline
from scclone2dr.trainer import Trainer, GuideType

# 1) Build the pipeline from the already-loaded real-data module.
trainer = Trainer(guide_type=GuideType.NONE)
pipeline = scClone2DRPipeline(
    data_source=datamodule,
    trainer=trainer,
    mode_nu="noise_correction",
    mode_theta="not shared decoupled",
)

# Ensure model topology is configured from the dataset metadata.
pipeline.model.configure(datamodule)

# 2) Fit on the already prepared data_train dictionary.
params_svi = pipeline.fit(
    data=data_train,
    penalty_l1=0.1,
    penalty_l2=0.1,
    lr=0.01,
    n_steps=2000,
)

# 3) Save learned parameters.
ckpt_dir = Path("./checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / "simu_data_pipeline_run.npz"
pipeline.save(ckpt_path)

# 4) Load trained pipeline from disk.
pipeline_loaded = scClone2DRPipeline.from_file(
    ckpt_path,
    data_source=datamodule,
)

# 5) Sample from posterior (Monte Carlo).
posterior_results = pipeline_loaded.sample_posterior(
    data=data_test,
    idxs_sample_eval=idxs_test,
    nb_ites=100,
    dir_save=None,
    sample_names=idxs_test,
    model_name="simu_data_",
)

evaluations = pipeline_loaded.evaluate(data_test, posterior_results['params'], true_params=gt_params_test)

# 6) Optional posterior predictive sample from the generative model.
posterior_predictive_data, _ = pipeline_loaded.model.sampling(data_test, params=posterior_results['params'])

print(f"Saved checkpoint: {ckpt_path}")

## True vs Predicted Plot
Scatter plot of true versus predicted fold changes.

In [ ]:
plt.scatter(evaluations.fold_change_true, evaluations.fold_change_pred)

## Observed vs Predicted Plot
Scatter plot of observed versus predicted fold changes.

In [ ]:
plt.scatter(evaluations.fold_change_data, evaluations.fold_change_pred)

## Simulated Tumour Fractions
Visualize the tumor fractions in control wells.

In [ ]:
scclone2dr.plots.show_fractions(data_test, posterior_results['data'], idxdrug=0)

## Simulated Cell Counts
Show predicted number of non-malignant cells in control wells.

In [ ]:
scclone2dr.plots.show_cells(data_test, posterior_results['data'])

## Simulated Proportions
Plot clone proportions for simulated data.

In [ ]:
scclone2dr.plots.show_proportions(data_test, posterior_results['params'])

## Simulated Beta Effects
Inspect beta parameters in the simulated setting.

In [ ]:
scclone2dr.plots.show_beta(data_test, posterior_results['params'], true_params=gt_params_test)

## Simulated Count Scatter
Visualize observed vs predicted counts for simulated data.

In [ ]:
scclone2dr.plots.scatter_counts(data_test, posterior_results['data'])

## Subclone Survival Probabilities
Compute survival probabilities from subclone-level features.

In [ ]:
scclone2dr.plots.survival_probabilities(data_test, posterior_results['params']['PI'], datamodule.cluster2clonelabel, idxdrug=0)